# G-Empirical Equivariance Deviation (G-EED) - Kvinge et al. (2022)

## 0. Setup

In [ ]:
import glob
import random
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image, ImageDraw
from torchvision import models
from torchvision.transforms import functional as TF
from tqdm import tqdm

RANDOM_STATE = 42
IMG_SIZE = 32
BACKGROUND_GRAY = 0.5
BATCH_SIZE = 64
N_SAMPLES = 50
DATA_ROOT = Path("../dataset/coco_crops_transparent_8cat")

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

IMAGENET_MEAN = models.VGG16_Weights.IMAGENET1K_V1.transforms().mean
IMAGENET_STD = models.VGG16_Weights.IMAGENET1K_V1.transforms().std


class PlainVGGFeatureExtractor(nn.Module):
    """VGG-16 features through layer 30 (inclusive) - 1x1x512 output at 32x32 input."""

    def __init__(self):
        super().__init__()
        weights = models.VGG16_Weights.IMAGENET1K_V1
        vgg = models.vgg16(weights=weights)
        self.features = nn.Sequential(*list(vgg.features.children())[:31])
        for p in self.parameters():
            p.requires_grad = False
        self.eval()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x)


feature_extractor = PlainVGGFeatureExtractor().to(device)
vgg_layers = list(feature_extractor.features.children())
print(feature_extractor)


In [ ]:
def apply_circular_aperture(img: Image.Image, background_gray: float) -> Image.Image:
    """Mask everything outside the image's inscribed circle to the gray background."""
    w, h = img.size
    radius = min(w, h) // 2
    cx, cy = w // 2, h // 2
    mask = Image.new("L", (w, h), 0)
    ImageDraw.Draw(mask).ellipse((cx - radius, cy - radius, cx + radius, cy + radius), fill=255)
    background = Image.new("RGB", (w, h), tuple(int(255 * background_gray) for _ in range(3)))
    return Image.composite(img, background, mask)


def load_rotated_rgba_on_gray(path: Path, angle: float, size: int = IMG_SIZE) -> torch.Tensor:
    """Rotate at original resolution, composite onto gray, circular aperture, resize, normalize."""
    img = Image.open(path).convert("RGBA")
    img = img.rotate(angle, resample=Image.BILINEAR, expand=False)
    background = Image.new("RGB", img.size, tuple(int(255 * BACKGROUND_GRAY) for _ in range(3)))
    background.paste(img, mask=img.split()[3])
    background = apply_circular_aperture(background, BACKGROUND_GRAY)
    background = background.resize((size, size), Image.BICUBIC)
    tensor = TF.to_tensor(background)
    tensor = TF.normalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
    return tensor


# Deterministic, sorted file list - then a fixed, seeded N_SAMPLES subsample, shared by both parts below.
file_list_full = sorted(str(p) for p in DATA_ROOT.glob("*/*.png"))
print(f"Found {len(file_list_full)} images across {len(set(Path(p).parent.name for p in file_list_full))} categories")

sample_rng = np.random.default_rng(RANDOM_STATE)
sample_idx = sample_rng.choice(len(file_list_full), size=N_SAMPLES, replace=False)
file_list = [file_list_full[i] for i in sorted(sample_idx)]
print(f"Using N_SAMPLES={len(file_list)} images (seeded subsample)")


def extract_features(angle: float) -> np.ndarray:
    """Batch-extract 512-dim VGG features (layer 30) for every image in file_list, at rotation `angle`."""
    features = []
    with torch.no_grad():
        for start in tqdm(range(0, len(file_list), BATCH_SIZE), desc=f"angle={angle}", leave=False):
            batch_paths = file_list[start:start + BATCH_SIZE]
            batch = torch.stack([load_rotated_rgba_on_gray(Path(p), angle) for p in batch_paths]).to(device)
            feats = feature_extractor(batch)
            features.append(feats.view(feats.size(0), -1).cpu().numpy())
            del batch, feats
    return np.concatenate(features, axis=0)


def build_intermediate(l: int) -> nn.Module:
    return nn.Sequential(*vgg_layers[:l + 1]).to(device).eval()


def extract_layer_features(l: int, angle: float) -> np.ndarray:
    """Batch-extract (N, C, H, W) feature maps at VGG layer l, at rotation `angle`."""
    intermediate = build_intermediate(l)
    features = []
    with torch.no_grad():
        for start in tqdm(range(0, len(file_list), BATCH_SIZE), desc=f"layer={l} angle={angle}", leave=False):
            batch_paths = file_list[start:start + BATCH_SIZE]
            batch = torch.stack([load_rotated_rgba_on_gray(Path(p), angle) for p in batch_paths]).to(device)
            feats = intermediate(batch)
            features.append(feats.cpu().numpy())
            del batch, feats
    return np.concatenate(features, axis=0)


## 1. G-EED Metric (Kvinge et al., 2022)

In [ ]:
def neg_cosine(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Negative-cosine distance between rows of a and b, shape (N, D) -> (N,).
    Defined as 1 - cos_sim (not -cos_sim) so identical vectors give distance 0."""
    a_t = torch.from_numpy(a).float()
    b_t = torch.from_numpy(b).float()
    return 1.0 - torch.nn.functional.cosine_similarity(a_t, b_t, dim=1).numpy()


def orbit_mean(feats_by_g: dict) -> np.ndarray:
    """f_hat(x) for the invariance case: mean feature vector over the full orbit {f(gx) : g in G}."""
    stacked = np.stack(list(feats_by_g.values()), axis=0)  # (|G|, N, D)
    return stacked.mean(axis=0)


def mean_pairwise_distance(ref_feats: np.ndarray, m) -> float:
    """M: mean distance between distinct pairs of images' reference (g=id) features."""
    n = ref_feats.shape[0]
    idx_a, idx_b = zip(*combinations(range(n), 2))
    idx_a, idx_b = np.array(idx_a), np.array(idx_b)
    return float(np.mean(m(ref_feats[idx_a], ref_feats[idx_b])))


def geed_scores(feats_by_g: dict, id_key, m) -> dict:
    """General G-EED, both instantiations, for one distance function m.
    feats_by_g: {g: (N, D) features for f(gx)}, id_key: the identity element's key in feats_by_g."""
    ref_feats = feats_by_g[id_key]
    f_hat_inv = orbit_mean(feats_by_g)
    M = mean_pairwise_distance(ref_feats, m)

    equiv_terms, inv_terms = [], []
    for g, feats_g in feats_by_g.items():
        equiv_terms.append(m(feats_g, ref_feats))
        inv_terms.append(m(feats_g, f_hat_inv))

    E_equiv = float(np.mean(np.concatenate(equiv_terms)))
    E_inv = float(np.mean(np.concatenate(inv_terms)))
    E_latent = E_inv / M
    return {"E_equiv": E_equiv, "E_inv": E_inv, "E_latent": E_latent, "M": M}


## 2. Part A: Last Layer, Full Rotation Sweep

`G` = `angles = list(range(-170, 181, 10))` (36 angles, `0°` is the identity element), applied
to layer-30 features (512-dim) for the fixed `N_SAMPLES=50` sample. For each angle we build the
orbit `{f(g x)}` incrementally (all angles up to and including the current one - the running
dict accumulates each angle's already-cached features) and evaluate `geed_scores` against that
partial-orbit `f_hat`, mirroring how `vgg_biscione_invariance.ipynb` sweeps `T(theta)`/`U(theta)`
per angle. Since the invariance target `f_hat(x)` is defined over the *whole* group, we first
extract features for every angle, then compute all scores against the complete orbit.

In [ ]:
angles = list(range(-170, 181, 10))
cardinal_angles = [-90, 0, 90, 180]

feats_by_angle = {}
for angle in tqdm(angles, desc="Extracting layer-30 features"):
    feats_by_angle[angle] = extract_features(angle)

assert feats_by_angle[0].shape[0] == N_SAMPLES

geed_summary_df = pd.DataFrame([{"distance": "cosine", **geed_scores(feats_by_angle, id_key=0, m=neg_cosine)}])
geed_summary_df


In [ ]:
def geed_per_angle(feats_by_g: dict, id_key, m) -> pd.DataFrame:
    """Per-angle G-EED summands, using the full-orbit f_hat (fixed across angles) as target."""
    ref_feats = feats_by_g[id_key]
    f_hat_inv = orbit_mean(feats_by_g)
    M = mean_pairwise_distance(ref_feats, m)

    rows = []
    for g, feats_g in feats_by_g.items():
        E_equiv = float(np.mean(m(feats_g, ref_feats)))
        E_inv = float(np.mean(m(feats_g, f_hat_inv)))
        rows.append({"angle": g, "E_equiv": E_equiv, "E_inv": E_inv, "E_latent": E_inv / M})
    return pd.DataFrame(rows).sort_values("angle").reset_index(drop=True)


geed_angle_df = geed_per_angle(feats_by_angle, id_key=0, m=neg_cosine)

zero_row = geed_angle_df.loc[geed_angle_df["angle"] == 0].iloc[0]
assert np.isclose(zero_row["E_equiv"], 0.0, atol=1e-5), (
    "E_equiv(0deg) is not ~0 - image correspondence is broken"
)

geed_angle_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(geed_angle_df["angle"], geed_angle_df["E_equiv"], marker="o", color="tab:red", label="E_equiv(theta)")
ax.plot(geed_angle_df["angle"], geed_angle_df["E_inv"], marker="s", color="tab:blue", label="E_inv(theta)")
ax.plot(geed_angle_df["angle"], geed_angle_df["E_latent"], marker="^", color="tab:green", label="E_latent(theta)")
for ca in cardinal_angles:
    ax.axvline(ca, color="red", linestyle=":", alpha=0.4)
ax.set_xlabel("Rotation angle (deg)")
ax.set_ylabel("G-EED score")
ax.set_xticks(angles)
ax.tick_params(axis="x", rotation=90)
ax.legend(fontsize=8)
ax.set_title("G-EED, layer 30, m = cosine")
plt.tight_layout()
plt.show()


## 3. Part B: Layerwise, One Fixed Transformation

In [ ]:
LAYER_INDICES = list(range(31))
ROTATION_ANGLE = 90


def compute_layerwise_geed(angle: float) -> pd.DataFrame:
    """Per-layer G-EED (cosine distance only - L2 is not meaningfully comparable across layers
    with wildly different feature scales/dimensions, see Section 1) for G = {0deg, angle}."""
    rows = []
    for l in tqdm(LAYER_INDICES, desc=f"Layerwise G-EED (angle={angle})"):
        X0_l = extract_layer_features(l, 0)
        Y_l = extract_layer_features(l, angle)
        n, c, h, w = X0_l.shape
        d_l = c * h * w
        feats_by_g = {0: X0_l.reshape(n, d_l), angle: Y_l.reshape(n, d_l)}

        row = {"layer": l, "d": d_l, "spatial": f"{c}x{h}x{w}"}
        scores = geed_scores(feats_by_g, id_key=0, m=neg_cosine)
        for k, v in scores.items():
            row[f"{k}_cosine"] = v
        rows.append(row)

    return pd.DataFrame(rows).sort_values("layer").reset_index(drop=True)


geed_layerwise_df = compute_layerwise_geed(ROTATION_ANGLE)
geed_layerwise_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(geed_layerwise_df["layer"], geed_layerwise_df["E_equiv_cosine"], marker="o", color="tab:red", label="E_equiv")
ax.plot(geed_layerwise_df["layer"], geed_layerwise_df["E_inv_cosine"], marker="s", color="tab:blue", label="E_inv")
ax.plot(geed_layerwise_df["layer"], geed_layerwise_df["E_latent_cosine"], marker="^", color="tab:green", label="E_latent")
ax.set_xlabel("VGG layer index")
ax.set_ylabel("G-EED score")
ax.set_xticks(geed_layerwise_df["layer"])
ax.legend(fontsize=8)
ax.set_title(f"Layerwise G-EED, {ROTATION_ANGLE}° rotation, m = cosine")
plt.tight_layout()
plt.show()
